In [1]:
# ===== [KAGGLE] bootstrap: recreate the Colab layout =====
# The paper notebooks expect /content/drive/MyDrive/bsm_paper_replication and
# /content/BSM_Project_2. bsm_env builds both from the attached dataset; the
# pipeline itself (images, augmentation, splits, training) is unchanged.
import pathlib, sys
_code = pathlib.Path("/kaggle/input/bsm-paper-2-kaggle/code")
if not _code.is_dir():
    _hits = list(pathlib.Path("/kaggle/input").rglob("bsm_env.py"))
    _code = _hits[0].parent if _hits else _code
if str(_code) not in sys.path:
    sys.path.insert(0, str(_code))
import bsm_env
BSM_RAW_SRC = bsm_env.setup()


[bsm_env] using existing /content
[bsm_env] mode=existing  project=/content/drive/MyDrive/bsm_paper_replication
[bsm_env] raw images=367  originals=367
[bsm_env] models.py: ok
[bsm_env] common.py: ok
[bsm_env] torch 2.11.0+cu128 cuda=True Tesla T4
[bsm_env] disk /kaggle/working free=20.9 GB


# BSM Paper Replication — Notebook 07: ResNet-34 (paper-faithful, real run)

SMOKE_TEST=False. 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3. Best-val checkpoint. Evaluate full test split. Stop if train_acc < 0.30 at epoch 30.

**Param count note:** `resnet34(head='paper')` uses Table 5 head (avgpool -> FC(feature, 6)) = **21,797,672** params (matches verified architecture).

In [2]:
# setup
import os, sys
# [kaggle] Colab Drive mount removed - bsm_env recreates the folder tree
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
import torch, random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SMOKE_TEST = False
print("gpu:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "SMOKE_TEST=", SMOKE_TEST)

gpu: True Tesla T4 SMOKE_TEST= False


## Real run: ResNet-34, 60 epochs, full-batch grad-accum (micro 32, one optim step/epoch), Adam 1e-3, best-val checkpoint

In [3]:
# ===== REAL TRAINING: ResNet-34 (60 epochs, paper-faithful) =====
import os, sys, time, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "resnet34"
RES  = pathlib.Path(PROJECT) / "results"
CKPT.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)

from models import resnet34
import common

# ---- data (split arrays from nb04) ----
def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    Xt = torch.from_numpy(X.transpose(0, 3, 1, 2)).contiguous()  # NHWC -> NCHW
    yt = torch.from_numpy(y.astype(np.int64))
    return Xt, yt

Xtr, ytr = load_xy("train")
Xva, yva = load_xy("val")
print("train:", Xtr.shape, "val:", Xva.shape, "| dtype", Xtr.dtype, "range", float(Xtr.min()), float(Xtr.max()))

MICRO = 32
train_loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=MICRO, shuffle=False)
val_loader   = DataLoader(TensorDataset(Xva, yva), batch_size=MICRO, shuffle=False)

# ---- model ----
torch.manual_seed(42)
model = resnet34().cuda()
n_params = sum(p.numel() for p in model.parameters())
print("resnet34 params:", f"{n_params:,}")

EPOCHS = 60
t0 = time.time()
hist = common.train_model(
    model, train_loader, val_loader, torch.device("cuda"),
    epochs=EPOCHS, full_batch=True,
    ckpt_dir=CKPT, history_path=RES / "history_resnet34.json",
    tag="resnet34", lr=1e-3, resume=True,
)
print(f"TRAINING DONE in {(time.time()-t0)/60:.1f} min")
print("best epoch:", hist.get("best_epoch"), "best val:", hist.get("best_val_acc"))

tr_acc = hist.get("train_acc") or []
if len(tr_acc) >= 30 and tr_acc[29] < 0.30:
    print("WARNING: train acc at epoch 30 =", tr_acc[29], "< 0.30 (spec stop-check)")

train: torch.Size([5651, 3, 224, 224]) val: torch.Size([1211, 3, 224, 224]) | dtype torch.float32 range 0.0 1.0
resnet34 params: 21,287,750
resnet34 epoch 01/60  train_loss 1.9099 acc 0.1793  val_loss 9600.7973 acc 0.1610  (16.2s)
resnet34 epoch 02/60  train_loss 3.5488 acc 0.2550  val_loss 106.3337 acc 0.1652  (16.5s)
resnet34 epoch 03/60  train_loss 2.2951 acc 0.2835  val_loss 6.6363 acc 0.2213  (16.7s)
resnet34 epoch 04/60  train_loss 2.5542 acc 0.2001  val_loss 546.7456 acc 0.1734  (15.8s)
resnet34 epoch 05/60  train_loss 2.0200 acc 0.2665  val_loss 246.7154 acc 0.1874  (15.4s)
resnet34 epoch 06/60  train_loss 1.7790 acc 0.3038  val_loss 55.5595 acc 0.1916  (15.4s)
resnet34 epoch 07/60  train_loss 1.6336 acc 0.2991  val_loss 3.0932 acc 0.2857  (15.7s)
resnet34 epoch 08/60  train_loss 1.5441 acc 0.3555  val_loss 6.0552 acc 0.2139  (15.9s)
resnet34 epoch 09/60  train_loss 1.4931 acc 0.3918  val_loss 4.2739 acc 0.2651  (15.8s)
resnet34 epoch 10/60  train_loss 1.4369 acc 0.4136  val_lo

### Re-engineering round (DEVIATIONS row 14)

If the train-vs-validation gap at the best-validation epoch exceeds 0.15, ONE
extra round is trained from a fresh model with lr 5e-4 and weight decay 1e-4, and
the round with the higher validation accuracy is the one evaluated on the test
split. The threshold and the round-1 hyper-parameters are our documented choice
(the paper does not define them).

In [4]:
# --- RE-ENGINEERING ROUND (spec: re-engineering if train - val gap > 0.15) ---
# Paper's Fig. 5 "model re-engineering" loop. The 0.15 threshold and the
# round-1 hyper-parameters (lr 5e-4, weight decay 1e-4) are OUR documented
# choice -- the paper does not define them (DEVIATIONS row 14).
import json as _json
import numpy as _np
import torch as _torch

RE_ENGINEER_GAP = 0.15
RE_ENGINEER_LR, RE_ENGINEER_WD = 5e-4, 1e-4
RE_ENGINEER_EPOCHS = 60

def _round_stats(tag, history_file):
    h = _json.load(open(history_file))
    b = int(_np.argmax(h["val_acc"])) + 1
    return {"tag": tag, "best_epoch": b,
            "best_val": float(h["val_acc"][b - 1]),
            "train_at_best": float(h["train_acc"][b - 1]),
            "train_seconds": float(_np.sum(h.get("epoch_time_s", []))),
            "gap": float(h["train_acc"][b - 1] - h["val_acc"][b - 1])}

ROUNDS = [_round_stats("resnet34", RES / "history_resnet34.json")]
print("round 0:", ROUNDS[0])

if ROUNDS[0]["gap"] > RE_ENGINEER_GAP:
    print(f"gap {ROUNDS[0]['gap']:+.4f} > {RE_ENGINEER_GAP} -> RE-ENGINEERING ROUND 1 "
          f"(fresh model, lr {RE_ENGINEER_LR}, wd {RE_ENGINEER_WD}, {RE_ENGINEER_EPOCHS} epochs)")
    _torch.manual_seed(SEED)
    model_re1 = resnet34().cuda()
    common.train_model(
        model_re1, train_loader, val_loader, _torch.device("cuda"),
        epochs=RE_ENGINEER_EPOCHS, full_batch=True,
        ckpt_dir=CKPT, history_path=RES / "history_resnet34_re1.json",
        tag="resnet34_re1", lr=RE_ENGINEER_LR, wd=RE_ENGINEER_WD, resume=True,
    )
    ROUNDS.append(_round_stats("resnet34_re1", RES / "history_resnet34_re1.json"))
    print("round 1:", ROUNDS[1])
else:
    print(f"gap {ROUNDS[0]['gap']:+.4f} <= {RE_ENGINEER_GAP} -> no re-engineering round needed")

SELECTED_ROUND = max(ROUNDS, key=lambda r: r["best_val"])
print("SELECTED ROUND (higher validation accuracy):", SELECTED_ROUND)
with open(RES / "selected_round_resnet34.json", "w") as f:
    _json.dump({"rounds": ROUNDS, "selected": SELECTED_ROUND}, f, indent=2)


round 0: {'tag': 'resnet34', 'best_epoch': 48, 'best_val': 0.7654830718414534, 'train_at_best': 0.7754379755795434, 'train_seconds': 939.4266874790192, 'gap': 0.009954903738090048}
gap +0.0100 <= 0.15 -> no re-engineering round needed
SELECTED ROUND (higher validation accuracy): {'tag': 'resnet34', 'best_epoch': 48, 'best_val': 0.7654830718414534, 'train_at_best': 0.7754379755795434, 'train_seconds': 939.4266874790192, 'gap': 0.009954903738090048}


In [5]:
# ===== EVAL ResNet-34 (best-val ckpt) on TEST SPLIT =====
import os, sys, json, pathlib
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
VER = pathlib.Path(PROJECT) / "versions"
CKPT = pathlib.Path(PROJECT) / "checkpoints" / "resnet34"
RES  = pathlib.Path(PROJECT) / "results"
RES.mkdir(parents=True, exist_ok=True)
if PROJECT not in sys.path: sys.path.insert(0, PROJECT)
from models import resnet34
import common

CLASSES = ["Ilmenite","Rutile","Zircon","Garnet","Monazite","Sillimanite"]

def load_xy(prefix):
    X = np.load(VER / f"X_{prefix}.npy")
    y = np.load(VER / f"y_{prefix}.npy")
    return torch.from_numpy(X.transpose(0,3,1,2)).contiguous(), torch.from_numpy(y.astype(np.int64))

Xte, yte = load_xy("test")
print("test:", Xte.shape)
test_loader = DataLoader(TensorDataset(Xte, yte), batch_size=32, shuffle=False)

device = torch.device("cuda")
model = resnet34().to(device)
print("evaluating selected round:", SELECTED_ROUND)
ck = torch.load(CKPT / f"{SELECTED_ROUND['tag']}_best.pt", map_location=device, weights_only=True)
model.load_state_dict(ck)
_ck = {"epoch": SELECTED_ROUND["best_epoch"]}
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_true.append(yb.numpy())
y_pred = np.concatenate(all_preds); y_true = np.concatenate(all_true)

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(6))
spec_list = []
for i in range(6):
    tn = cm.sum() - cm[i,:].sum() - cm[:,i].sum() + cm[i,i]
    fp = cm[:,i].sum() - cm[i,i]
    spec_list.append(tn/(tn+fp) if (tn+fp)>0 else 0.0)
spec = float(np.mean(spec_list))

print(f"\n=== ResNet-34 TEST SPLIT (best epoch {_ck['epoch']}) ===")
print(f"accuracy {acc:.4f} | macro precision {prec:.4f} | macro recall {rec:.4f} | macro F1 {f1:.4f} | macro specificity {spec:.4f}")
print("\nconfusion matrix (rows=true):")
print(cm)
print("\nper-class:")
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4, zero_division=0))

results = {
  "model": "resnet34", "notebook": "07_resnet34",
  "best_epoch": int(_ck.get("epoch", -1)),
  "selected_round": SELECTED_ROUND,
  "re_engineering": ROUNDS,
  "train_minutes": round(sum(r.get("train_seconds", 0.0) for r in ROUNDS) / 60, 2),
  "test_split": {"accuracy": float(acc), "macro_precision": float(prec), "macro_recall": float(rec),
                 "macro_f1": float(f1), "macro_specificity": spec,
                 "confusion_matrix": cm.tolist(), "n_test": int(len(y_true))},
  "paper_reported_table9": {"accuracy": 0.83, "precision": 0.89, "recall": 0.83, "note": "PAPER numbers (Table 9), for comparison only"},
}
with open(RES / "metrics_resnet34.json", "w") as f:
    json.dump(results, f, indent=2)
print("\nsaved", RES / "metrics_resnet34.json")
print("EVAL DONE")

test: torch.Size([1212, 3, 224, 224])
evaluating selected round: {'tag': 'resnet34', 'best_epoch': 48, 'best_val': 0.7654830718414534, 'train_at_best': 0.7754379755795434, 'train_seconds': 939.4266874790192, 'gap': 0.009954903738090048}

=== ResNet-34 TEST SPLIT (best epoch 48) ===
accuracy 0.7673 | macro precision 0.7836 | macro recall 0.7704 | macro F1 0.7645 | macro specificity 0.9536

confusion matrix (rows=true):
[[189   0   7   5   0   4]
 [ 40 119   5   4  29  21]
 [  6   1 170   7   4   7]
 [  8   1  10 158   6  15]
 [  0   0  35  13 148   2]
 [  4   3  14   1  30 146]]

per-class:
              precision    recall  f1-score   support

    Ilmenite     0.7652    0.9220    0.8363       205
      Rutile     0.9597    0.5459    0.6959       218
      Zircon     0.7054    0.8718    0.7798       195
      Garnet     0.8404    0.7980    0.8187       198
    Monazite     0.6820    0.7475    0.7133       198
 Sillimanite     0.7487    0.7374    0.7430       198

    accuracy           

In [6]:
# persist notebook copy
import glob, os, shutil
PROJECT = "/content/drive/MyDrive/bsm_paper_replication"
os.makedirs(os.path.join(PROJECT,"notebooks"), exist_ok=True)
cands = glob.glob("/content/drive/MyDrive/Colab Notebooks/*.ipynb")
if cands:
    newest = max(cands, key=os.path.getmtime)
    shutil.copy(newest, os.path.join(PROJECT,"notebooks","07_resnet34.ipynb"))
    print("copied ->", os.path.join(PROJECT,"notebooks","07_resnet34.ipynb"))